## CONFIG

In [ ]:
import re, unicodedata, difflib

## List of true labels in Italian
true_labels = [
    "arrivare","andare","aspettare","avere","capire","chiamare","chiedere","conoscere","dare","dire","dovere",
    "essere","fare","mettere","potere","prendere","sapere","sentire","trovare","venire","aprire","chiudere",
    "mangiare","bere","accendere","spegnere","volere","bene","si","no","più","poco","molto","sempre","adesso",
    "poi","male","sopra","sotto","destra","sinistra","avanti","indietro","oggi","domani","ieri","forse","prima",
    "perchè","anche","come","però","quindi","quando","dove","se","oppure","io","lui","lei","noi","voi","loro",
    "tu","questo","quello","buono","bello","cattivo","brutto","grande","piccolo","nuovo","vecchio","cosa","parte",
    "anno","casa","problema","aiuto","tempo","lavoro","persona","acqua","cibo","bisogno","donna","uomo","gruppo",
    "guerra","idea","macchina","mano","oggetto","telefono","computer","domanda","uno","mille","paura","ansia",
    "gioia","felicità","tristezza","serenità","amore","morte","bagno","dolore","riposo"
]

def _strip_diacritics(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def _robust_decode(x) -> str:
    if isinstance(x, (bytes, bytearray)):
        for enc in ('utf-8', 'latin-1', 'cp1252'):
            try:
                s = x.decode(enc)
                break
            except Exception:
                continue
    else:
        s = str(x)

    s = s.replace('\x00', '').strip()

    if ('Ã' in s) or ('Â' in s):
        try:
            s = s.encode('latin-1', 'ignore').decode('utf-8', 'ignore')
        except Exception:
            pass

    s = unicodedata.normalize('NFC', s)
    return s

def canonicalize_label(raw) -> str:
    s = _robust_decode(raw).lower()

    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'_img$', '', s)

    s = (s.replace('’', "'")
           .replace('‘', "'")
           .replace('“', '"')
           .replace('”', '"'))
    s = s.replace("''", "'").replace("´", "'").replace("`", "'").strip()

    fixes = {
        "perche": "perchè", "perche'": "perchè", "perch'e": "perchè", "perchà''": "perchè",
        "pero": "però", "piu": "più", "serenita": "serenità", "felicita": "felicità"
    }
    if s in fixes: 
        return fixes[s]

    if s in true_labels:
        return s

    s_ascii = _strip_diacritics(s)
    candidates_ascii = [_strip_diacritics(t) for t in true_labels]

    match1 = difflib.get_close_matches(s_ascii, candidates_ascii, n=1, cutoff=0.7)
    if match1:
        idx = candidates_ascii.index(match1[0])
        return true_labels[idx]

    match2 = difflib.get_close_matches(s, true_labels, n=1, cutoff=0.6)
    if match2:
        return match2[0]

    return s


In [ ]:
import h5py
import pandas as pd
from pathlib import Path
import json

def build_eeg_dataframe(h5_dir, label_map_path):
    with open(label_map_path, "r", encoding="utf-8") as f:
        label2idx = json.load(f)

    rows = []
    for file in sorted(Path(h5_dir).glob("*.h5")):
        subject_id, session_id = file.stem.split("_")
        with h5py.File(file, "r") as f:
            data = f["data"]
            labels = f["labels"][:]
            n_epochs, n_channels, n_samples = data.shape

            for i, lbl_raw in enumerate(labels):
                label_name = canonicalize_label(lbl_raw)
                label_idx = label2idx.get(label_name, -1)
                rows.append({
                    "subject_id": subject_id,
                    "session_id": session_id,
                    "epoch_idx": i,
                    "label_name": label_name,
                    "label_idx": label_idx,
                    "n_channels": n_channels,
                    "n_samples": n_samples,
                    "fs": 256,
                    "path_h5": str(file)
                })

    df = pd.DataFrame(rows)
    return df


## 🧠 EEG Intelligent DataFrame – Overview

This DataFrame acts as the **central index** for your entire EEG dataset.  
Instead of duplicating raw data, it keeps **structured metadata** that describes every EEG epoch — including where it’s stored, which subject/session it belongs to, and its semantic label.

---

### 📁 Structure

Each row of the DataFrame corresponds to **one EEG epoch** and contains:

| Column | Description |
|:--------|:-------------|
| `subject_id` | ID of the participant (e.g. `11`) |
| `session_id` | Recording session (e.g. `S002`) |
| `epoch_idx` | Index of the epoch inside the `.h5` file |
| `label_name` | Semantic label (e.g. *"felicità"*, *"paura"*) |
| `label_idx` | Numerical class index from `label2idx.json` |
| `n_channels` | Number of EEG channels in that epoch |
| `n_samples` | Number of samples per channel |
| `path_h5` | Absolute path to the `.h5` file containing the signal |

---

### ⚙️ Purpose

The **EEG Intelligent DataFrame** serves as a lightweight, queryable "map" of your dataset.  
It allows you to:

1. **Explore** dataset composition and class balance.  
2. **Filter and retrieve** EEG signals on demand (by subject, session, or label).  
3. **Attach new features** (e.g., spectral power, connectivity metrics).  
4. **Build higher-level datasets** for PyTorch or PyTorch Geometric (graphs).  
5. **Visualize** or debug specific epochs without reloading everything.



In [ ]:
from pathlib import Path

project_root = Path.cwd().parents[0] 

h5_dir = project_root / "data" / "processed"
label_map_path = project_root / "data" / "interim" / "label2idx.json"
output_csv = project_root / "data" / "interim" / "eeg_metadata.csv"

meta_df_eeg = build_eeg_dataframe(
    h5_dir=str(h5_dir),
    label_map_path=str(label_map_path)
)

meta_df_eeg.to_csv(output_csv, index=False)

meta_df_eeg


,subject_id,session_id,epoch_idx,label_name,label_idx,n_channels,n_samples,fs,path_h5
0,11,S001,0,cibo,84,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
1,11,S001,1,aprire,20,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
2,11,S001,2,mettere,13,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
3,11,S001,3,nuovo,72,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
4,11,S001,4,bello,67,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
...,...,...,...,...,...,...,...,...,...
1089,11,S005,215,avere,3,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
1090,11,S005,216,mangiare,22,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
1091,11,S005,217,cosa,74,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...
1092,11,S005,218,nuovo,72,61,384,256,/Users/dani/Documents/Thesis_ImaSpe/Daniele_IS...


In [27]:
import pandas as pd
# Number of occurrences of each label
print(meta_df_eeg["label_name"].value_counts().head(10))


label_name
cibo        10
acqua       10
spegnere    10
dolore      10
trovare     10
sempre      10
avere       10
prendere    10
quando      10
potere      10
Name: count, dtype: int64


In [28]:
# Number of epochs per subject
print(meta_df_eeg.groupby("subject_id")["epoch_idx"].count())


subject_id
11    1094
Name: epoch_idx, dtype: int64


In [29]:
# Classes present in each subject
print(meta_df_eeg.groupby("subject_id")["label_name"].nunique())


subject_id
11    110
Name: label_name, dtype: int64


In [30]:
# Percentage of each class
print(meta_df_eeg["label_name"].value_counts(normalize=True) * 100)


label_name
cibo        0.914077
acqua       0.914077
spegnere    0.914077
dolore      0.914077
trovare     0.914077
              ...   
se          0.914077
prima       0.914077
uno         0.731261
si          0.731261
telefono    0.731261
Name: proportion, Length: 110, dtype: float64
